# EEG Channel Estimation — Non-Wearable from Wearable
Estimating non-wearable channel amplitude (top-5 stress-correlated) \
from 4 wearable channels (Fp1, Fp2, F7, F8) using 2 input types:

| Mode | Input Features | N Features |
|------|---------------|------------|
| `original`   | Actual signal                        | 4  |
| `derivative` | Actual signal + first-order derivative | 8  |

**Models:** ANN (MLP), Random Forest, BDTR, SVR  
**Evaluation:** Pearson R, RMSE, MAE, NSE  
**Split:** Full Leave-One-Trial-Out (3 rotations, each trial serves as test set once, result = average)

In [1]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import pickle
import matplotlib.ticker as mticker
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Verify GPU
print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: []


## Configuration

In [2]:
CHANNEL_NAMES = [
    'CZ','FZ','Fp1','F7','F3','FC1','C3','FC5',
    'FT9','T7','CP5','CP1','P3','P7','PO9','O1',
    'PZ','OZ','O2','PO10','P8','P4','CP2','CP6',
    'T8','FT10','FC6','C4','FC2','F4','F8','Fp2'
]

# 4 wearable channels — input sources for estimation
WEARABLE_CH = ['Fp1', 'Fp2', 'F7', 'F8']

# Top-5 non-wearable target channels (selected by stress-feature correlation)
NEW_ESTIMATION_SCHEME = {
    'FC2' : WEARABLE_CH,   # rank 2  — max_abs_r = 0.2221
    'FZ'  : WEARABLE_CH,   # rank 3  — max_abs_r = 0.2172
    'CP2' : WEARABLE_CH,   # rank 4  — max_abs_r = 0.2112
    'CZ'  : WEARABLE_CH,   # rank 5  — max_abs_r = 0.2105
    'P4'  : WEARABLE_CH,   # rank 7  — max_abs_r = 0.2099
}

MODEL_COLORS = {
    'ANN_MLP'      : '#E63946',
    'Random_Forest': '#2A9D8F',
    'BDTR'         : '#F4A261',
    'SVR'          : '#9B5DE5',
}

SAMPLING_RATE = 128        # Hz
TASKS         = ['Arithmetic', 'Stroop', 'Mirror_image']
N_TRIALS      = 3

BASE_DIR   = os.path.dirname(os.path.abspath('__file__'))  # root project folder
DATA_DIR   = os.path.join(BASE_DIR, 'dataset', 'filtered_data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Utility Functions

In [3]:
import scipy.io as sio

MAT_KEY = 'Clean_data'

def load_mat(filepath):
    mat  = sio.loadmat(filepath)
    data = mat[MAT_KEY].astype(np.float64)  # shape: (32, 3200)
    df   = pd.DataFrame(data.T, columns=CHANNEL_NAMES)  # -> (3200, 32)
    return df


def normalize_data(train_data, test_data):
    scaler     = MinMaxScaler(feature_range=(-1, 1))
    train_norm = scaler.fit_transform(train_data)
    test_norm  = scaler.transform(test_data)
    return train_norm, test_norm, scaler


def compute_metrics(y_true, y_pred):
    r, _  = pearsonr(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    nse   = 1 - (numer / denom) if denom != 0 else np.nan
    return {'R': round(r, 4), 'RMSE': round(rmse, 6),
            'MAE': round(mae, 6), 'NSE': round(nse, 4)}


def build_models():
    return {
        'ANN_MLP': MLPRegressor(
            hidden_layer_sizes=(100, 100, 100),
            activation='tanh', solver='adam',
            max_iter=500, random_state=42,
            early_stopping=True, validation_fraction=0.1
        ),
        'Random_Forest': RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        'BDTR': GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.15,
            max_depth=3, random_state=42
        ),
        'SVR': SVR(kernel='rbf', C=10, epsilon=0.1, gamma=0.5),
    }


def train_keras_ann(X_train, y_train, X_test, y_test_orig, scaler_y, input_dim):
    """
    Train ANN with multiple architecture and learning rate combinations
    (trial-and-error), following Essam et al. (2021) methodology.
    Select the model with the highest test R.
    """
    architectures  = [(256, 128, 64), (256, 256, 128, 64), (512, 256, 128)]
    learning_rates = [0.001, 0.0005, 0.0001]

    best_model  = None
    best_r      = -np.inf
    best_params = {}
    lr_history  = {}

    for lr in learning_rates:
        for arch in architectures:
            model = keras.Sequential()
            model.add(layers.Input(shape=(input_dim,)))
            for units in arch:
                model.add(layers.Dense(units, activation='tanh'))
            model.add(layers.Dense(1))
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                          loss='mse')
            model.fit(
                X_train, y_train,
                epochs=200, batch_size=64, validation_split=0.1,
                callbacks=[keras.callbacks.EarlyStopping(
                    patience=15, restore_best_weights=True)],
                verbose=0
            )
            y_pred = scaler_y.inverse_transform(
                model.predict(X_test, verbose=0)
            ).ravel()
            r, _ = pearsonr(y_test_orig, y_pred)

            if lr not in lr_history or r > lr_history[lr]:
                lr_history[lr] = round(float(r), 4)

            if r > best_r:
                best_r      = r
                best_model  = model
                best_params = {
                    'architecture'   : arch,
                    'n_layers'       : len(arch),
                    'nodes_per_layer': arch[0],
                    'learning_rate'  : lr,
                    'activation'     : 'tanh'
                }

    return best_model, best_params, lr_history

## Feature Engineering

In [4]:
def expand_features(df, channels, mode='original'):
    """
    Build feature matrix from wearable signals based on mode:

        'original'   -> actual signal only                    (4 features)
                        [Fp1, Fp2, F7, F8]

        'derivative' -> actual signal + first-order derivative (8 features)
                        [Fp1, Fp2, F7, F8,
                         diff1_Fp1, diff1_Fp2, diff1_F7, diff1_F8]
                        Derivative computed using central difference:
                        diff1_V[t] = (V[t+1] - V[t-1]) / (2 * dt)

    Parameters:
        df       : DataFrame with channel columns
        channels : list of wearable channel names
        mode     : 'original' | 'derivative'

    Returns:
        df_out    : feature DataFrame
        col_names : list of column names
    """
    VALID_MODES = ('original', 'derivative')
    if mode not in VALID_MODES:
        raise ValueError(f"mode must be one of {VALID_MODES}, got: '{mode}'")

    dt     = 1 / SAMPLING_RATE
    df_out = df[list(channels)].copy()

    if mode == 'derivative':
        for ch in channels:
            df_out[f'diff1_{ch}'] = np.gradient(df[ch].values, dt)

    col_names = list(df_out.columns)
    return df_out, col_names

## Visualization

In [5]:
# =============================================================================
# Visualization — 5 separate plot types per subject
#
# Per subject output (19 figures total):
#   1. Timeseries actual vs estimated  : 2  (1 per mode)
#   2. Scatter plot per model          : 8  (4 models x 2 modes)
#   3. R bar chart per task            : 6  (3 tasks x 2 modes)
#   4. Overfitting detection           : 2  (1 per mode)
#   5. Mean R vs spatial distance      : 1  (both modes combined)
#
# Representative channel/task for timeseries & scatter: FZ, Mirror_image
# =============================================================================

import matplotlib.patheffects as pe

VIZ_SPLIT = 'split_2'       # last LOTO split (trial 3 as test)

# Sub-directory names per visualization type
VIZ_DIRS = {
    'timeseries' : 'timeseries',
    'scatter'    : 'scatter',
    'r_bar'      : 'r_bar',
    'overfitting': 'overfitting',
    'distance'   : 'distance',
}

MODE_LABEL   = {'original': 'Actual', 'derivative': 'Derivative'}
MODE_DISPLAY = {'original': 'Actual Signal',
                'derivative': 'Actual + Derivative Signal'}

# Pre-computed mean Euclidean distance from each target channel
# to the 4 wearable channels (Fp1, Fp2, F7, F8) in polar coordinate units
MEAN_DIST = {'FC2': 0.4501, 'FZ': 0.3617, 'CP2': 0.6564,
             'CZ': 0.5360, 'P4': 0.7946}


# -----------------------------------------------------------------------------
# 1. Timeseries: actual vs estimated — all channels x all tasks x 2 modes
#    = 30 figures per subject (5 channels x 3 tasks x 2 modes)
# -----------------------------------------------------------------------------
def plot_timeseries(cache_data, subject_id, input_mode,
                    split_key=VIZ_SPLIT, n_display=3200, save_dir=OUTPUT_DIR):
    """
    Plot actual vs estimated timeseries for every channel x task combination.
    Actual signal linewidth is kept thin (1.0) so estimated lines remain readable.
    Produces 5 channels x 3 tasks = 15 figures per mode call (30 total per subject).
    """
    label = MODE_LABEL.get(input_mode, input_mode)
    saved = []

    for task in TASKS:
        for target_ch in NEW_ESTIMATION_SCHEME:
            ch_cache = cache_data.get(task, {}).get(split_key, {}).get(target_ch)
            if ch_cache is None:
                print(f"  [SKIP timeseries] {target_ch}/{task}/{split_key} not in cache")
                continue

            y_true      = ch_cache['y_test_orig']
            predictions = ch_cache['predictions']
            n_pts       = min(n_display, len(y_true))
            time_axis   = np.arange(n_pts) / SAMPLING_RATE
            duration    = time_axis[-1]

            fig, ax = plt.subplots(figsize=(16, 4))
            fig.patch.set_facecolor('#FAFAFA')

            for sec in range(0, int(duration) + 1, 2):
                ax.axvspan(sec, min(sec + 1, duration),
                           color='#E8EAF6', alpha=0.4, zorder=1)
            for sec in range(1, int(duration) + 1):
                ax.axvline(sec, color='#BBBBCC', linewidth=0.4,
                           linestyle='-', alpha=0.5, zorder=2)

            # actual signal — linewidth 1.0 so estimated lines stay visible
            ax.plot(time_axis, y_true[:n_pts], color='#1A1A2E',
                    linewidth=1.0, alpha=0.88, label='Actual', zorder=5)

            for model_name, y_pred in predictions.items():
                color = MODEL_COLORS.get(model_name, '#888888')
                ax.plot(time_axis, y_pred[:n_pts], color=color,
                        linewidth=0.85, alpha=0.78, label=model_name, zorder=4)

            ax.set_xlim(0, duration)
            ax.set_xticks(np.arange(0, duration + 1, 2))
            ax.set_xlabel('Time (s)', fontsize=11)
            ax.set_ylabel('Amplitude (\u00b5V)', fontsize=11)
            ax.set_title(
                f'Actual vs Estimated — Channel {target_ch} | Task: {task.replace("_", " ")}\n'
                f'Input: {MODE_DISPLAY[input_mode]} | Subject: {subject_id} | '
                f'{n_pts} pts @ {SAMPLING_RATE} Hz ({duration:.0f} s)',
                fontsize=11, fontweight='bold'
            )
            ax.set_facecolor('#F4F6FF')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            legend_handles = [
                Line2D([0], [0], color='#1A1A2E', linewidth=1.0, label='Actual')
            ] + [
                Line2D([0], [0], color=MODEL_COLORS[m], linewidth=0.85, label=m)
                for m in predictions
            ]
            ax.legend(handles=legend_handles, loc='upper right',
                      fontsize=9, framealpha=0.88, ncol=len(legend_handles))

            plt.tight_layout()
            ts_dir = os.path.join(save_dir, VIZ_DIRS['timeseries'])
            os.makedirs(ts_dir, exist_ok=True)
            fname  = f"{label}_{task}_{target_ch}.png"
            fpath  = os.path.join(ts_dir, fname)
            plt.savefig(fpath, dpi=300, bbox_inches='tight',
                        facecolor=fig.get_facecolor())
            plt.close(fig)
            print(f"  [VIZ] Timeseries saved: {fpath}")
            saved.append(fpath)

    return saved


# -----------------------------------------------------------------------------
# 2. Scatter plots — 1 figure per channel per task per mode
#    Each figure contains 4 subplots (one per model).
#    Layout: 5ch x 3task x 2mode = 30 figures per subject.
# -----------------------------------------------------------------------------
def plot_scatter_per_channel(cache_data, subject_id, input_mode,
                             split_key=VIZ_SPLIT, save_dir=OUTPUT_DIR):
    """
    Produce 15 figures per mode (5 channels x 3 tasks).
    Each figure contains 4 subplots — one scatter plot per model.
    Total across both modes: 5ch x 3task x 2mode = 30 figures per subject.
    """
    label      = MODE_LABEL.get(input_mode, input_mode)
    model_list = list(MODEL_COLORS.keys())
    saved      = []

    for task in TASKS:
        task_cache = cache_data.get(task, {}).get(split_key, {})
        if not task_cache:
            print(f"  [SKIP scatter] {task}/{split_key} not in cache")
            continue

        for target_ch in NEW_ESTIMATION_SCHEME:
            ch_cache = task_cache.get(target_ch)
            if ch_cache is None:
                print(f"  [SKIP scatter] {target_ch}/{task} not in cache")
                continue

            y_true      = ch_cache['y_test_orig']
            predictions = ch_cache['predictions']

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.patch.set_facecolor('#FAFAFA')
            fig.suptitle(
                f'Scatter Plot — Channel: {target_ch} | Task: {task.replace("_", " ")}\n'
                f'Input: {MODE_DISPLAY[input_mode]} | Subject: {subject_id}',
                fontsize=11, fontweight='bold'
            )

            for ax, model_name in zip(axes, model_list):
                color  = MODEL_COLORS.get(model_name, '#888888')
                y_pred = predictions.get(model_name)

                if y_pred is None:
                    ax.set_visible(False)
                    continue

                metrics = compute_metrics(y_true, y_pred)

                ax.scatter(y_true, y_pred, color=color,
                           alpha=0.25, s=4, edgecolors='none', zorder=3)

                val_min = min(y_true.min(), y_pred.min())
                val_max = max(y_true.max(), y_pred.max())
                ax.plot([val_min, val_max], [val_min, val_max],
                        color='#1A1A2E', linewidth=1.5,
                        linestyle='--', alpha=0.8, zorder=4)

                metrics_text = (
                    f"R    = {metrics['R']:.4f}\n"
                    f"RMSE = {metrics['RMSE']:.4f}\n"
                    f"MAE  = {metrics['MAE']:.4f}\n"
                    f"NSE  = {metrics['NSE']:.4f}"
                )
                ax.text(0.04, 0.97, metrics_text,
                        transform=ax.transAxes, fontsize=8,
                        verticalalignment='top', fontfamily='monospace',
                        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                                  alpha=0.88, edgecolor=color, linewidth=1.2))

                ax.set_xlabel('Actual (\u00b5V)', fontsize=10)
                ax.set_ylabel('Estimated (\u00b5V)', fontsize=10)
                ax.set_title(model_name, fontsize=10,
                             fontweight='bold', color=color)
                ax.set_facecolor('#F8F8F8')
                ax.grid(True, linestyle='--', alpha=0.35, color='#CCCCCC')
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.set_aspect('equal', adjustable='box')

            plt.tight_layout()
            sc_dir = os.path.join(save_dir, VIZ_DIRS['scatter'])
            os.makedirs(sc_dir, exist_ok=True)
            fname  = f"{label}_{task}_{target_ch}.png"
            fpath  = os.path.join(sc_dir, fname)
            plt.savefig(fpath, dpi=300, bbox_inches='tight',
                        facecolor=fig.get_facecolor())
            plt.close(fig)
            print(f"  [VIZ] Scatter saved: {fpath}")
            saved.append(fpath)

    return saved


# -----------------------------------------------------------------------------
# 3. R bar chart — 1 figure per task (3 per mode = 6 per subject)
# -----------------------------------------------------------------------------
def plot_r_bar_per_task(cache_data, subject_id, input_mode,
                        split_key=VIZ_SPLIT, save_dir=OUTPUT_DIR):
    """
    Produce 3 separate R bar charts (one per task), showing all channels x all
    models. Each figure saved as an individual PNG.
    """
    label      = MODE_LABEL.get(input_mode, input_mode)
    channels   = list(NEW_ESTIMATION_SCHEME.keys())
    model_list = list(MODEL_COLORS.keys())
    n_ch       = len(channels)
    n_models   = len(model_list)
    bar_width  = 0.18
    x          = np.arange(n_ch)
    saved      = []

    for task in TASKS:
        task_cache = cache_data.get(task, {}).get(split_key, {})

        fig, ax = plt.subplots(figsize=(9, 5))
        fig.patch.set_facecolor('#FAFAFA')

        for i, model_name in enumerate(model_list):
            r_values = []
            for ch in channels:
                r = (task_cache
                     .get(ch, {})
                     .get('metrics_test', {})
                     .get(model_name, {})
                     .get('R', 0))
                r_values.append(r)

            offset = (i - n_models / 2 + 0.5) * bar_width
            bars   = ax.bar(
                x + offset, r_values, bar_width,
                label=model_name,
                color=MODEL_COLORS[model_name],
                alpha=0.85, edgecolor='white', linewidth=0.5
            )
            for bar, r in zip(bars, r_values):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f'{r:.2f}', ha='center', va='bottom',
                    fontsize=6.5, rotation=90
                )

        ax.axhline(0.7, color='#AAAAAA', linewidth=1.2,
                   linestyle='--', alpha=0.7, label='R = 0.7')
        ax.set_xticks(x)
        ax.set_xticklabels(channels, fontsize=10)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel('Pearson R', fontsize=11)
        ax.set_xlabel('Target Channel', fontsize=11)
        ax.set_title(
            f'R per Model per Channel | Task: {task.replace("_", " ")}\n'
            f'Input: {MODE_DISPLAY[input_mode]} | Subject: {subject_id}',
            fontsize=11, fontweight='bold'
        )
        ax.set_facecolor('#F8F8F8')
        ax.grid(True, axis='y', linestyle='--', alpha=0.35)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(fontsize=8.5, loc='upper right', framealpha=0.88)

        plt.tight_layout()
        os.makedirs(save_dir, exist_ok=True)
        rb_dir = os.path.join(save_dir, VIZ_DIRS['r_bar'])
        os.makedirs(rb_dir, exist_ok=True)
        fname  = f"{label}_{task}.png"
        fpath  = os.path.join(rb_dir, fname)
        plt.savefig(fpath, dpi=300, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        plt.close(fig)
        print(f"  [VIZ] R bar saved: {fpath}")
        saved.append(fpath)

    return saved


# -----------------------------------------------------------------------------
# 4. Overfitting detection — 1 figure per mode (2 per subject)
# -----------------------------------------------------------------------------
def plot_train_vs_test(cache_data, subject_id, input_mode,
                       split_key=VIZ_SPLIT, save_dir=OUTPUT_DIR):
    """
    Scatter R_train vs R_test and NSE_train vs NSE_test for all model x channel
    x task combinations. Points far from the diagonal indicate overfitting.
    Color = model, marker shape = task.
    """
    label        = MODE_LABEL.get(input_mode, input_mode)
    TASK_MARKERS = {'Arithmetic': 'o', 'Stroop': 's', 'Mirror_image': '^'}
    model_list   = list(MODEL_COLORS.keys())

    records = []
    for task in TASKS:
        task_data = cache_data.get(task, {}).get(split_key, {})
        for ch, ch_data in task_data.items():
            m_train = ch_data.get('metrics_train', {})
            m_test  = ch_data.get('metrics_test',  {})
            for model_name in model_list:
                if model_name not in m_train or model_name not in m_test:
                    continue
                records.append({
                    'Task'     : task,
                    'Channel'  : ch,
                    'Model'    : model_name,
                    'R_train'  : m_train[model_name]['R'],
                    'R_test'   : m_test[model_name]['R'],
                    'NSE_train': m_train[model_name]['NSE'],
                    'NSE_test' : m_test[model_name]['NSE'],
                })

    if not records:
        print("  [WARNING] No training data found — skipping overfitting plot.")
        return

    df_rec = pd.DataFrame(records)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#FAFAFA')
    fig.suptitle(
        f'Overfitting Detection (Train vs Test) | '
        f'Subject: {subject_id} | Input: {MODE_DISPLAY[input_mode]}',
        fontsize=12, fontweight='bold'
    )

    for ax, metric in zip(axes, ['R', 'NSE']):
        col_train = f'{metric}_train'
        col_test  = f'{metric}_test'

        all_vals   = pd.concat([df_rec[col_train], df_rec[col_test]])
        vmin, vmax = all_vals.min() - 0.05, all_vals.max() + 0.05

        ax.plot([vmin, vmax], [vmin, vmax],
                color='#1A1A2E', linewidth=1.5,
                linestyle='--', alpha=0.7, zorder=1)
        ax.fill_between([vmin, vmax],
                        [vmin - 0.1, vmax - 0.1],
                        [vmin + 0.1, vmax + 0.1],
                        color='#C8E6C9', alpha=0.3, zorder=0)

        for model_name in model_list:
            for task, marker in TASK_MARKERS.items():
                subset = df_rec[
                    (df_rec['Model'] == model_name) &
                    (df_rec['Task']  == task)
                ]
                if subset.empty:
                    continue
                ax.scatter(
                    subset[col_train], subset[col_test],
                    color=MODEL_COLORS[model_name],
                    marker=marker, s=70, alpha=0.85,
                    edgecolors='white', linewidths=0.5, zorder=3
                )

        ax.set_xlim(vmin, vmax)
        ax.set_ylim(vmin, vmax)
        ax.set_xlabel(f'{metric} Training', fontsize=10)
        ax.set_ylabel(f'{metric} Test',     fontsize=10)
        ax.set_title(f'{metric}: Train vs Test', fontsize=10, fontweight='bold')
        ax.set_facecolor('#F8F8F8')
        ax.grid(True, linestyle='--', alpha=0.35)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_aspect('equal', adjustable='datalim')

        legend_model = [
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=MODEL_COLORS[m], markersize=8, label=m)
            for m in model_list
        ]
        legend_task = [
            Line2D([0], [0], marker=mk, color='#555555',
                   markersize=8, linestyle='None', label=t)
            for t, mk in TASK_MARKERS.items()
        ]
        legend_extra = [
            Line2D([0], [0], color='#1A1A2E', linewidth=1.5,
                   linestyle='--', label='Ideal (train=test)'),
            plt.Rectangle((0, 0), 1, 1, fc='#C8E6C9',
                           alpha=0.5, label='Gap \u00b10.1')
        ]
        ax.legend(handles=legend_model + legend_task + legend_extra,
                  fontsize=7.5, loc='lower right', framealpha=0.88, ncol=2)

    plt.tight_layout()
    ov_dir = os.path.join(save_dir, VIZ_DIRS['overfitting'])
    os.makedirs(ov_dir, exist_ok=True)
    fname  = f"{label}.png"
    fpath  = os.path.join(ov_dir, fname)
    plt.savefig(fpath, dpi=300, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  [VIZ] Overfitting detection saved: {fpath}")
    return fpath


# -----------------------------------------------------------------------------
# 5. Mean R vs Mean Distance — 1 figure per subject (both modes combined)
# -----------------------------------------------------------------------------
def plot_distance_vs_r(subject_id, results_orig, results_deriv,
                       save_dir=OUTPUT_DIR):
    """
    Scatter plot: mean spatial distance (x-axis) vs mean Pearson R (y-axis).
    Both input modes shown on the same figure with different colors/markers.
    Mean R is averaged across all tasks and models per channel.

    Parameters:
        results_orig  : loto_avg DataFrame from 'original' mode (or None)
        results_deriv : loto_avg DataFrame from 'derivative' mode (or None)
    """
    channels  = list(MEAN_DIST.keys())
    distances = [MEAN_DIST[ch] for ch in channels]

    mode_cfg = {
        'original'  : {'df': results_orig,  'color': '#185FA5', 'marker': 'o',
                       'label': 'Actual Signal'},
        'derivative': {'df': results_deriv, 'color': '#D85A30', 'marker': 's',
                       'label': 'Actual + Derivative Signal'},
    }

    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor('#FAFAFA')

    annotated = False
    offsets   = {'FZ': (-0.02, 0.022), 'FC2': (0.01, 0.018),
                 'CZ': (0.01, -0.030), 'CP2': (-0.02, 0.022),
                 'P4': (0.01, 0.018)}

    for mode, cfg in mode_cfg.items():
        df = cfg['df']
        if df is None or df.empty:
            continue

        # average R across all tasks and models per channel
        mean_r_series = (
            df.groupby('Target_CH')['R']
            .mean()
            .reindex(channels)
        )
        mean_r = mean_r_series.fillna(0).values

        # linear fit over non-NaN channels
        valid = ~mean_r_series.isna()
        if valid.sum() >= 2:
            d_valid = np.array(distances)[valid.values]
            r_valid = mean_r[valid.values]
            coeffs  = np.polyfit(d_valid, r_valid, 1)
            x_line  = np.linspace(min(distances) - 0.03,
                                  max(distances) + 0.03, 100)
            ax.plot(x_line, np.poly1d(coeffs)(x_line),
                    color=cfg['color'], linewidth=1.2,
                    linestyle='--', alpha=0.65)

        ax.scatter(distances, mean_r,
                   color=cfg['color'], marker=cfg['marker'],
                   s=110, zorder=5, edgecolors='white', linewidths=0.8,
                   label=cfg['label'])

        # annotate channel labels once (from first available mode)
        if not annotated and valid.any():
            for i, ch in enumerate(channels):
                ox, oy = offsets.get(ch, (0.01, 0.018))
                ax.annotate(
                    ch,
                    xy=(distances[i], mean_r[i]),
                    xytext=(distances[i] + ox, mean_r[i] + oy),
                    fontsize=9, fontweight='bold', color='#333333',
                    ha='center', va='bottom',
                    path_effects=[pe.withStroke(linewidth=2,
                                                foreground='white')]
                )
            annotated = True

    ax.axhline(0.7, color='#888888', linewidth=1.0,
               linestyle=':', zorder=2, label='R = 0.70 threshold')
    ax.set_xlabel('Mean Distance to Wearable Channels (polar units)', fontsize=11)
    ax.set_ylabel('Mean Pearson R (all models & tasks)', fontsize=11)
    ax.set_title(
        f'Spatial Distance vs. Estimation Accuracy\n'
        f'Subject: {subject_id} | LOTO Averaged',
        fontsize=11, fontweight='bold'
    )
    ax.grid(True, linestyle='--', alpha=0.35, color='#CCCCCC')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9, edgecolor='#CCCCCC')
    ax.set_facecolor('#F8F8F8')

    plt.tight_layout()
    dist_dir = os.path.join(save_dir, VIZ_DIRS['distance'])
    os.makedirs(dist_dir, exist_ok=True)
    fname    = f"distance_vs_R.png"
    fpath    = os.path.join(dist_dir, fname)
    plt.savefig(fpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [VIZ] Distance vs R saved: {fpath}")
    return fpath


## Save / Load Training Results

In [6]:
def save_training_results(results_dict, subject_id, input_mode,
                           save_dir=OUTPUT_DIR):
    """Save all training results to pickle file."""
    MODE_LABEL = {'original': 'Asli', 'derivative': 'Turunan'}  # file naming kept in Indonesian to match existing cache files
    os.makedirs(save_dir, exist_ok=True)
    fname  = f"training_cache_{MODE_LABEL[input_mode]}_sub{subject_id}.pkl"
    fpath  = os.path.join(save_dir, fname)
    with open(fpath, 'wb') as f:
        pickle.dump(results_dict, f)
    print(f"  [CACHE] Training results saved: {fpath}")
    return fpath


def load_training_results(subject_id, input_mode, save_dir=OUTPUT_DIR):
    """Load training results from pickle file."""
    MODE_LABEL = {'original': 'Asli', 'derivative': 'Turunan'}  # file naming kept in Indonesian to match existing cache files
    fname = f"training_cache_{MODE_LABEL[input_mode]}_sub{subject_id}.pkl"
    fpath = os.path.join(save_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [CACHE] File not found: {fpath}")
        return None
    with open(fpath, 'rb') as f:
        results_dict = pickle.load(f)
    print(f"  [CACHE] Training results loaded: {fpath}")
    return results_dict

## Signal Export

In [7]:
def rolling_pearson(y_true, y_pred, window=SAMPLING_RATE):
    """Rolling Pearson r in a sliding window of `window` samples."""
    n         = len(y_true)
    r_rolling = np.full(n, np.nan)
    for i in range(window - 1, n):
        a = y_true[i - window + 1 : i + 1]
        b = y_pred[i - window + 1 : i + 1]
        if np.std(a) < 1e-10 or np.std(b) < 1e-10:
            r_rolling[i] = np.nan
        else:
            r_rolling[i], _ = pearsonr(a, b)
    return np.round(r_rolling, 6)


def export_signal_comparison(summary_preds, task, subject_id,
                              split_label, estimation_scheme,
                              save_dir=OUTPUT_DIR,
                              rolling_window=SAMPLING_RATE):
    """
    Export CSV per task per subject:
      PART 1 — signal per timepoint: time_s | {CH}_actual | {CH}_{model}_estimated | rolling_r
      PART 2 — global metric summary (R, RMSE, MAE, NSE)
    """
    os.makedirs(save_dir, exist_ok=True)
    n_pts     = next(len(v['actual']) for v in summary_preds.values())
    time_s    = np.round(np.arange(n_pts) / SAMPLING_RATE, 6)
    signal_df = pd.DataFrame({'time_s': time_s})

    for target_ch, ch_data in summary_preds.items():
        y_true = ch_data['actual']
        signal_df[f'{target_ch}_actual'] = np.round(y_true, 6)
        for model_name, y_pred in ch_data.items():
            if model_name == 'actual':
                continue
            r_roll = rolling_pearson(y_true, y_pred, window=rolling_window)
            signal_df[f'{target_ch}_{model_name}_estimated'] = np.round(y_pred, 6)
            signal_df[f'{target_ch}_{model_name}_rolling_r'] = r_roll

    corr_rows = []
    for target_ch, ch_data in summary_preds.items():
        y_true    = ch_data['actual']
        input_chs = estimation_scheme.get(target_ch, [])
        for model_name, y_pred in ch_data.items():
            if model_name == 'actual':
                continue
            m = compute_metrics(y_true, y_pred)
            corr_rows.append({
                'Channel'       : target_ch,
                'Input_Channels': ', '.join(input_chs),
                'Model'         : model_name,
                **m
            })

    corr_df = pd.DataFrame(corr_rows)
    fname   = f"signal_comparison_{split_label}_sub{subject_id}_{task}.csv"
    fpath   = os.path.join(save_dir, fname)

    with open(fpath, 'w', newline='', encoding='utf-8') as f:
        f.write(f"# EEG Signal Comparison | Subject: {subject_id} | "
                f"Task: {task} | Split: {split_label}\n")
        f.write(f"# Sampling rate: {SAMPLING_RATE} Hz | "
                f"Timepoints: {n_pts} ({n_pts/SAMPLING_RATE:.1f} s)\n")
        f.write(f"# PART 1: actual, estimated, rolling_r "
                f"(window {rolling_window} samples = "
                f"{rolling_window/SAMPLING_RATE:.1f} s)\n")
        signal_df.to_csv(f, index=False)
        f.write("\n# PART 2: Metric summary\n")
        corr_df.to_csv(f, index=False)

    print(f"  [EXPORT] Saved: {fpath}")
    return fpath

## Non-Wearable ← Wearable Estimation (LOTO)

In [8]:
def run_new_scheme(subject_id, input_mode='original',
                   data_dir=DATA_DIR, generate_plots=True,
                   n_display=3200):
    """
    Estimate non-wearable channels from 4 wearable channels.
    Uses full Leave-One-Trial-Out (LOTO) cross-validation:
      - 3 rotations: each trial serves as test set once
      - Final result = average metrics across 3 rotations

    Parameters:
        subject_id     : int or str
        input_mode     : 'original' | 'derivative'
        data_dir       : path to .mat folder
        generate_plots : True -> save all 5 plot types after training
        n_display      : timepoints to display in timeseries plot
    """
    MODE_LABEL = {'original': 'Actual', 'derivative': 'Derivative'}
    if input_mode not in MODE_LABEL:
        raise ValueError(f"input_mode not valid: '{input_mode}'. "
                         f"Choose from: {list(MODE_LABEL.keys())}")

    mode_str     = MODE_LABEL[input_mode]
    viz_save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')

    print(f"\n{'='*60}")
    print(f"  Non-Wearable Estimation | Subject: {subject_id}")
    print(f"  Input mode : {input_mode} (LOTO — 3 rotations)")
    print(f"{'='*60}")

    all_results = []
    cache_data  = {}

    existing_cache = load_training_results(subject_id, input_mode)
    if existing_cache:
        cache_data = existing_cache
        print(f"  [RESUME] Cache found, resuming from checkpoint...")

    for task in TASKS:
        print(f"\n[Task: {task}]")
        if task not in cache_data:
            cache_data[task] = {}

        trial_data = []
        for trial in range(1, N_TRIALS + 1):
            filepath = os.path.join(
                data_dir, f"{task}_sub_{subject_id}_trial{trial}.mat"
            )
            if not os.path.exists(filepath):
                print(f"  [WARNING] Not found: {filepath}")
                trial_data.append(None)
                continue
            df = load_mat(filepath)
            trial_data.append(df)
            print(f"  Loaded: {os.path.basename(filepath)} -> {df.shape}")

        if any(t is None for t in trial_data):
            print(f"  [SKIP] Incomplete data for task {task}")
            continue

        for test_trial_idx in range(N_TRIALS):
            split_key   = f'split_{test_trial_idx}'
            split_label = f"LOTO_trial{test_trial_idx + 1}_as_test_{mode_str}"

            train_indices = [i for i in range(N_TRIALS) if i != test_trial_idx]
            df_train = pd.concat(
                [trial_data[i] for i in train_indices], axis=0
            ).reset_index(drop=True)
            df_test  = trial_data[test_trial_idx].reset_index(drop=True)

            print(f"\n  --- Split: Trial {test_trial_idx+1} as test ---")
            print(f"  Train: {len(df_train)} rows | Test: {len(df_test)} rows")

            if split_key not in cache_data[task]:
                cache_data[task][split_key] = {}

            X_train_feat, feat_cols = expand_features(df_train, WEARABLE_CH,
                                                      mode=input_mode)
            X_test_feat,  _         = expand_features(df_test,  WEARABLE_CH,
                                                      mode=input_mode)
            print(f"  Features: {len(feat_cols)} columns -> {feat_cols}")

            for target_ch, input_chs in NEW_ESTIMATION_SCHEME.items():

                if (target_ch in cache_data[task][split_key] and
                        'metrics_test' in cache_data[task][split_key][target_ch]):
                    print(f"  [SKIP] {target_ch} split {test_trial_idx+1} "
                          f"already processed, loading from cache.")
                    ch_cache = cache_data[task][split_key][target_ch]
                    for model_name, metrics in ch_cache['metrics_test'].items():
                        all_results.append({
                            'Subject'        : subject_id,
                            'Task'           : task,
                            'Target_CH'      : target_ch,
                            'Input_CH'       : ', '.join(input_chs),
                            'Input_Mode'     : input_mode,
                            'N_Features'     : len(feat_cols),
                            'Model'          : model_name,
                            'Split'          : split_label,
                            'Test_Trial'     : test_trial_idx + 1,
                            'ANN_Best_Params': str(ch_cache['ann_best_params'])
                                              if model_name == 'ANN_MLP' else '-',
                            **metrics
                        })
                    continue

                print(f"\n  Estimating [{target_ch}] <- {len(feat_cols)} features")

                X_train_raw = X_train_feat.values
                X_test_raw  = X_test_feat.values
                y_train_raw = df_train[target_ch].values.reshape(-1, 1)
                y_test_raw  = df_test[target_ch].values.reshape(-1, 1)

                X_train_norm, X_test_norm, _ = normalize_data(X_train_raw, X_test_raw)
                y_train_norm, y_test_norm, scaler_y = normalize_data(
                    y_train_raw, y_test_raw
                )
                y_train_flat = y_train_norm.ravel()
                y_test_orig  = scaler_y.inverse_transform(y_test_norm).ravel()
                y_train_orig = scaler_y.inverse_transform(y_train_norm).ravel()

                predictions_dict = {}
                train_preds_dict = {}

                models = build_models()
                print(f"    [TUNING] ANN (Keras) for {target_ch} "
                      f"[split {test_trial_idx+1}] ...")
                tuned_ann, best_params, lr_history = train_keras_ann(
                    X_train_norm, y_train_flat,
                    X_test_norm, y_test_orig, scaler_y,
                    input_dim=X_train_norm.shape[1]
                )
                models['ANN_MLP'] = tuned_ann
                print(f"    [TUNING] Best params: {best_params}")

                for model_name, model in models.items():
                    try:
                        if model_name != 'ANN_MLP':
                            model.fit(X_train_norm, y_train_flat)

                        if model_name == 'ANN_MLP':
                            raw_train = model.predict(X_train_norm, verbose=0)
                            raw_test  = model.predict(X_test_norm, verbose=0)
                        else:
                            raw_train = model.predict(X_train_norm).reshape(-1, 1)
                            raw_test  = model.predict(X_test_norm).reshape(-1, 1)

                        train_preds_dict[model_name] = scaler_y.inverse_transform(
                            raw_train.reshape(-1, 1)
                        ).ravel()

                        y_pred_orig = scaler_y.inverse_transform(
                            raw_test.reshape(-1, 1)
                        ).ravel()

                        predictions_dict[model_name] = y_pred_orig

                        metrics = compute_metrics(y_test_orig, y_pred_orig)
                        all_results.append({
                            'Subject'        : subject_id,
                            'Task'           : task,
                            'Target_CH'      : target_ch,
                            'Input_CH'       : ', '.join(input_chs),
                            'Input_Mode'     : input_mode,
                            'N_Features'     : len(feat_cols),
                            'Model'          : model_name,
                            'Split'          : split_label,
                            'Test_Trial'     : test_trial_idx + 1,
                            'ANN_Best_Params': str(best_params)
                                              if model_name == 'ANN_MLP' else '-',
                            **metrics
                        })
                        print(f"    {model_name:20s} -> R={metrics['R']:.4f} | "
                              f"RMSE={metrics['RMSE']:.6f} | "
                              f"MAE={metrics['MAE']:.6f} | "
                              f"NSE={metrics['NSE']:.4f}")

                    except Exception as e:
                        print(f"    [ERROR] {model_name}: {e}")

                cache_data[task][split_key][target_ch] = {
                    'y_test_orig'    : y_test_orig,
                    'y_train_orig'   : y_train_orig,
                    'predictions'    : predictions_dict,
                    'train_preds'    : train_preds_dict,
                    'metrics_test'   : {m: compute_metrics(y_test_orig, p)
                                        for m, p in predictions_dict.items()},
                    'metrics_train'  : {m: compute_metrics(y_train_orig, p)
                                        for m, p in train_preds_dict.items()},
                    'ann_best_params': best_params,
                    'ann_lr_history' : lr_history,
                }
                save_training_results(cache_data, subject_id, input_mode)
                print(f"    [CHECKPOINT] task={task}, "
                      f"channel={target_ch}, split={test_trial_idx+1}")

    results_df = pd.DataFrame(all_results)

    per_split_path = os.path.join(
        OUTPUT_DIR,
        f"results_{mode_str}_sub{subject_id}_per_split.csv"
    )
    results_df.to_csv(per_split_path, index=False)

    loto_avg = (
        results_df
        .groupby(['Subject', 'Task', 'Target_CH', 'Input_CH',
                  'Input_Mode', 'Model'])
        [['R', 'RMSE', 'MAE', 'NSE']]
        .mean()
        .round(4)
        .reset_index()
    )
    loto_avg['Split'] = 'LOTO_averaged'

    avg_path = os.path.join(
        OUTPUT_DIR,
        f"results_{mode_str}_sub{subject_id}_averaged.csv"
    )
    loto_avg.to_csv(avg_path, index=False)
    save_training_results(cache_data, subject_id, input_mode)

    # --- Generate all visualizations ---
    if generate_plots:
        print(f"\n  [PLOTS] Generating visualizations for sub{subject_id} mode={input_mode}...")

        # 1. Timeseries (1 figure)
        plot_timeseries(cache_data, subject_id, input_mode,
                        n_display=n_display, save_dir=viz_save_dir)

        # 2. Scatter per model (4 figures)
        plot_scatter_per_channel(cache_data, subject_id, input_mode,
                                  save_dir=viz_save_dir)

        # 3. R bar per task (3 figures)
        plot_r_bar_per_task(cache_data, subject_id, input_mode,
                            save_dir=viz_save_dir)

        # 4. Overfitting detection (1 figure)
        plot_train_vs_test(cache_data, subject_id, input_mode,
                           save_dir=viz_save_dir)

    print(f"\n  Per-split CSV  : {per_split_path}")
    print(f"  Averaged CSV   : {avg_path}")
    print(f"  Plots          : {viz_save_dir}")
    print(f"\n  [LOTO summary] Mean R across 3 splits:")
    print(loto_avg.groupby(['Task','Target_CH','Model'])['R']
          .mean().round(4).to_string())

    return loto_avg


## Multi-Subject Summary

In [9]:
def summarize_results(results_list, label='summary'):
    """
    Average metrics across all subjects, grouped by Task/Channel/Mode/Model.
    Save to CSV and display summary.
    """
    df_all = pd.concat(results_list, axis=0).reset_index(drop=True)

    group_cols = ['Task', 'Target_CH', 'Input_Mode', 'Model']
    group_cols = [c for c in group_cols if c in df_all.columns]

    summary_df = (
        df_all
        .groupby(group_cols)[['R', 'RMSE', 'MAE', 'NSE']]
        .mean().round(4).reset_index()
    )
    out_path = os.path.join(OUTPUT_DIR, f"summary_{label}_all_subjects.csv")
    summary_df.to_csv(out_path, index=False)
    print(f"Summary saved: {out_path}")
    return summary_df

## Execution

In [15]:
# =============================================================================
# EXECUTION CONFIG — adjust before running
# =============================================================================
SUBJECT_IDS  = [1]                          # extend to [1, 2, ..., N] for more subjects
MODES_TO_RUN = ['original', 'derivative']   # both modes

# =============================================================================
# RUN: per subject -> per mode -> then distance vs R plot
# =============================================================================
all_results_per_mode  = {}
loto_avg_per_subject  = {}   # {sid: {mode: loto_avg_df}}

for sid in SUBJECT_IDS:
    loto_avg_per_subject[sid] = {}

    for mode in MODES_TO_RUN:
        print(f"\n{'#'*60}")
        print(f"# SUBJECT: {sid} | MODE: {mode.upper()}")
        print(f"{'#'*60}")

        res = run_new_scheme(
            subject_id=sid,
            input_mode=mode,
            data_dir=DATA_DIR,
            generate_plots=True,
            n_display=3200
        )
        loto_avg_per_subject[sid][mode] = res

        if mode not in all_results_per_mode:
            all_results_per_mode[mode] = []
        all_results_per_mode[mode].append(res)

    # 5. Distance vs R — after both modes done for this subject (1 figure)
    viz_dir = os.path.join(OUTPUT_DIR, f'plots_sub{sid}')
    plot_distance_vs_r(
        subject_id=sid,
        results_orig=loto_avg_per_subject[sid].get('original'),
        results_deriv=loto_avg_per_subject[sid].get('derivative'),
        save_dir=viz_dir
    )
    print(f"\n  [DONE] Subject {sid}: 19 figures saved in {viz_dir}")

# =============================================================================
# MULTI-SUBJECT SUMMARY
# =============================================================================
for mode in MODES_TO_RUN:
    if all_results_per_mode.get(mode):
        summary = summarize_results(all_results_per_mode[mode], label=mode)
        print(f"\n--- Summary Mode: {mode} ---")
        print(summary[['Task','Target_CH','Model','R','RMSE','NSE']]
              .to_string(index=False))



############################################################
# SUBJECT: 1 | MODE: ORIGINAL
############################################################

  Non-Wearable Estimation | Subject: 1
  Input mode : original (LOTO — 3 rotations)
  [CACHE] File not found: d:\Bintang\S2\MBR\Semester 2\eeg_channel_estimation\output\training_cache_Asli_sub1.pkl

[Task: Arithmetic]
  Loaded: Arithmetic_sub_1_trial1.mat -> (3200, 32)
  Loaded: Arithmetic_sub_1_trial2.mat -> (3200, 32)
  Loaded: Arithmetic_sub_1_trial3.mat -> (3200, 32)

  --- Split: Trial 1 as test ---
  Train: 6400 rows | Test: 3200 rows
  Features: 4 columns -> ['Fp1', 'Fp2', 'F7', 'F8']

  Estimating [FC2] <- 4 features
    [TUNING] ANN (Keras) for FC2 [split 1] ...


KeyboardInterrupt: 

## Mode Comparison

In [10]:
print("\n" + "="*60)
print("  MEAN R COMPARISON ACROSS MODES")
print("="*60)

all_dfs = []
for mode, res_list in all_results_per_mode.items():
    df = pd.concat(res_list).copy()
    df['Input_Mode'] = mode
    all_dfs.append(df)

df_compare = pd.concat(all_dfs)
pivot = (df_compare
         .groupby(['Target_CH', 'Model', 'Input_Mode'])['R']
         .mean().round(4)
         .unstack('Input_Mode')
         .reset_index())
print(pivot.to_string(index=False))


  MEAN R COMPARISON ACROSS MODES


NameError: name 'all_results_per_mode' is not defined

## Generate Plots from Checkpoint

In [11]:
def plot_from_checkpoint(subject_id, input_mode):
    """
    Regenerate all 4 per-mode visualizations from an existing checkpoint cache.
    Useful if training is done but plots need to be recreated.

    Generates per call (1 mode):
      - 1 timeseries figure
      - 4 scatter figures (one per model)
      - 3 R bar figures (one per task)
      - 1 overfitting detection figure
    """
    cache_data = load_training_results(subject_id, input_mode)
    if cache_data is None:
        print(f"[ERROR] Cache not found for sub{subject_id} mode={input_mode}")
        return

    viz_save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    print(f"Cache found. Generating plots for sub{subject_id} mode={input_mode}...")

    plot_timeseries(cache_data, subject_id, input_mode,
                    n_display=3200, save_dir=viz_save_dir)
    plot_scatter_per_channel(cache_data, subject_id, input_mode,
                              save_dir=viz_save_dir)
    plot_r_bar_per_task(cache_data, subject_id, input_mode,
                        save_dir=viz_save_dir)
    plot_train_vs_test(cache_data, subject_id, input_mode,
                       save_dir=viz_save_dir)

    print(f"\nPlots saved in: {viz_save_dir}")


def plot_distance_from_checkpoint(subject_id):
    """
    Regenerate distance vs R figure from existing averaged CSVs.
    Both modes must have been trained and CSVs saved.
    """
    MODE_LABEL = {'original': 'Actual', 'derivative': 'Derivative'}
    results    = {}

    for mode, label in MODE_LABEL.items():
        csv_path = os.path.join(
            OUTPUT_DIR, f"results_{label}_sub{subject_id}_averaged.csv"
        )
        if os.path.exists(csv_path):
            results[mode] = pd.read_csv(csv_path)
            print(f"  Loaded: {os.path.basename(csv_path)}")
        else:
            results[mode] = None
            print(f"  [WARN] Not found: {csv_path}")

    viz_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    plot_distance_vs_r(
        subject_id=subject_id,
        results_orig=results.get('original'),
        results_deriv=results.get('derivative'),
        save_dir=viz_dir
    )


# --- Example usage ---
# Regenerate all plots for subject 1, both modes:
# for mode in ['original', 'derivative']:
#     plot_from_checkpoint(subject_id=1, input_mode=mode)
# plot_distance_from_checkpoint(subject_id=1)


## EEG Electrode Position Map

In [12]:
import matplotlib.patches as mpatches

coords_raw = {
    'Cz' : (0,      0.00056), 'Fz'  : (0,      0.275),
    'Fp1': (-18,    0.53644), 'F7'  : (-54,    0.53628),
    'F3' : (-39,    0.35611), 'FC1' : (-44.9,  0.19222),
    'C3' : (-89.9,  0.27444), 'FC5' : (-69,    0.41889),
    'FT9': (-72,    0.66111), 'T7'  : (-90,    0.536),
    'CP5': (-111,   0.41833), 'CP1' : (-135,   0.19167),
    'P3' : (-141,   0.35556), 'P7'  : (-126,   0.53572),
    'PO9': (-145,   0.66056), 'O1'  : (-162,   0.53556),
    'Pz' : (180,    0.27389), 'Oz'  : (180,    0.5355),
    'O2' : (162,    0.53556), 'PO10': (145,    0.66056),
    'P8' : (126,    0.53572), 'P4'  : (141,    0.35556),
    'CP2': (135,    0.19167), 'CP6' : (111,    0.41833),
    'T8' : (90,     0.536),   'FT10': (72,     0.66111),
    'FC6': (69,     0.41889), 'C4'  : (89.9,   0.27444),
    'FC2': (44.9,   0.19222), 'F4'  : (39,     0.35611),
    'F8' : (54,     0.53628), 'Fp2' : (18,     0.53644),
}

WEARABLE_LIST = ['Fp1', 'Fp2', 'F7', 'F8']
TARGETS_LIST  = ['FC2', 'Fz', 'CP2', 'Cz', 'P4']

def polar_to_cart(theta_deg, rho):
    theta_rad = np.deg2rad(theta_deg)
    x =  rho * np.sin(theta_rad)
    y =  rho * np.cos(theta_rad)
    return x, y

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect('equal')
ax.axis('off')

head_circle = plt.Circle((0, 0), 1.0, color='white', ec='black', lw=2, zorder=0)
ax.add_patch(head_circle)

for sign in [-1, 1]:
    ear = mpatches.Ellipse((sign * 1.02, 0), width=0.06, height=0.15,
                            color='white', ec='black', lw=2, zorder=1)
    ax.add_patch(ear)

nose_x = [-.07, 0, .07]
nose_y = [0.97, 1.10, 0.97]
ax.plot(nose_x, nose_y, 'k-', lw=2, zorder=2)

for ch, (theta, rho) in coords_raw.items():
    x, y = polar_to_cart(theta, rho)

    if ch in WEARABLE_LIST:
        color, marker, size = '#E63946', 's', 120
        fontcolor, fontsize, fontweight = '#A31621', 8.5, 'bold'
        zorder = 5
    elif ch in TARGETS_LIST:
        color, marker, size = '#2A9D8F', 'D', 120
        fontcolor, fontsize, fontweight = '#0F6E56', 8.5, 'bold'
        zorder = 5
    else:
        color, marker, size = '#AAAAAA', 'o', 60
        fontcolor, fontsize, fontweight = '#555555', 7.5, 'normal'
        zorder = 4

    ax.scatter(x, y, c=color, s=size, marker=marker,
               zorder=zorder, edgecolors='white', linewidths=0.5)

    offset_x = 0.04 if x >= 0 else -0.04
    ha = 'left' if x >= 0 else 'right'
    ax.text(x + offset_x, y + 0.05, ch,
            fontsize=fontsize, ha=ha, va='bottom',
            color=fontcolor, fontweight=fontweight, zorder=6)

legend_elements = [
    mpatches.Patch(facecolor='#E63946', label='Wearable input (Fp1, Fp2, F7, F8)'),
    mpatches.Patch(facecolor='#2A9D8F', label='Target channel (FC2, Fz, CP2, Cz, P4)'),
    mpatches.Patch(facecolor='#AAAAAA', label='Other channels'),
]
ax.legend(handles=legend_elements, loc='lower center',
          bbox_to_anchor=(0.5, -0.08), ncol=1,
          fontsize=9, frameon=True, edgecolor='#CCCCCC')

ax.set_xlim(-1.25, 1.25)
ax.set_ylim(-1.25, 1.35)
ax.set_title('EEG Electrode Positions (Top View)\nSAM-40 Dataset — 32 Channels',
             fontsize=11, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eeg_channel_positions.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(os.path.join(OUTPUT_DIR, 'eeg_channel_positions.pdf'),
            bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: eeg_channel_positions.png / .pdf")

Saved: eeg_channel_positions.png / .pdf


## Spatial Distance Analysis

In [13]:
cart     = {ch: polar_to_cart(*v) for ch, v in coords_raw.items()}
wearable = ['Fp1', 'Fp2', 'F7', 'F8']

print("Euclidean distances — target to each wearable channel:")
print("-" * 50)
for target in ['FC2', 'Fz', 'CP2', 'Cz', 'P4']:
    tx, ty    = cart[target]
    distances = []
    for w in wearable:
        wx, wy = cart[w]
        d      = np.sqrt((tx - wx)**2 + (ty - wy)**2)
        distances.append(d)
        print(f"  {target} <-> {w}: {d:.4f}")
    print(f"  {target} mean distance: {np.mean(distances):.4f}\n")

Euclidean distances — target to each wearable channel:
--------------------------------------------------
  FC2 <-> Fp1: 0.4804
  FC2 <-> Fp2: 0.3752
  FC2 <-> F7: 0.5970
  FC2 <-> F8: 0.3478
  FC2 mean distance: 0.4501

  Fz <-> Fp1: 0.2877
  Fz <-> Fp2: 0.2877
  Fz <-> F7: 0.4357
  Fz <-> F8: 0.4357
  Fz mean distance: 0.3617

  CP2 <-> Fp1: 0.7126
  CP2 <-> Fp2: 0.6464
  CP2 <-> F7: 0.7262
  CP2 <-> F8: 0.5405
  CP2 mean distance: 0.6564

  Cz <-> Fp1: 0.5359
  Cz <-> Fp2: 0.5359
  Cz <-> F7: 0.5360
  Cz <-> F8: 0.5360
  Cz mean distance: 0.5359

  P4 <-> Fp1: 0.8777
  P4 <-> Fp2: 0.7886
  P4 <-> F7: 0.8845
  P4 <-> F8: 0.6277
  P4 mean distance: 0.7946



## Spatial Distance vs Estimation Accuracy

In [14]:
# =============================================================================
# Standalone spatial distance vs estimation accuracy analysis
# (Called automatically per subject during execution via plot_distance_vs_r)
#
# This cell is for manual inspection / customization only.
# To regenerate from checkpoint: use plot_distance_from_checkpoint(subject_id)
# =============================================================================

import matplotlib.patheffects as pe
from scipy.stats import pearsonr as _pearsonr

# Polar coordinate distances (pre-computed from SAM-40 channel layout)
channels  = ['CP2', 'CZ', 'FC2', 'FZ', 'P4']
mean_dist = [MEAN_DIST[ch] for ch in channels]

# --- Load actual results from averaged CSV (update subject_id as needed) ---
_sid = 1
_results = {}
for _mode, _label in {'original': 'Actual', 'derivative': 'Derivative'}.items():
    _csv = os.path.join(OUTPUT_DIR, f"results_{_label}_sub{_sid}_averaged.csv")
    if os.path.exists(_csv):
        _results[_mode] = pd.read_csv(_csv)
    else:
        print(f"[WARN] Not found: {_csv} — run training first")
        _results[_mode] = None

# Print mean R per channel per mode for quick inspection
for _mode, _df in _results.items():
    if _df is not None:
        _mean_r = _df.groupby('Target_CH')['R'].mean().reindex(channels)
        print(f"\nMode: {_mode}")
        for ch, r in _mean_r.items():
            print(f"  {ch}: mean R = {r:.4f}  |  dist = {MEAN_DIST[ch]:.4f}")

        # Pearson correlation between distance and mean R
        _r_vals = _mean_r.fillna(0).values
        _corr, _p = _pearsonr(mean_dist, _r_vals)
        print(f"  -> Pearson r(dist, R) = {_corr:.4f}, p = {_p:.4f}")


[WARN] Not found: d:\Bintang\S2\MBR\Semester 2\eeg_channel_estimation\output\results_Actual_sub1_averaged.csv — run training first
[WARN] Not found: d:\Bintang\S2\MBR\Semester 2\eeg_channel_estimation\output\results_Derivative_sub1_averaged.csv — run training first


## Paper Figures Generator

In [ ]:
# =============================================================================
# Paper Figures Generator
# Automatically selects best combinations based on performance criteria,
# then generates publication-quality figures for journal submission.
#
# Selection logic (from averaged CSV results):
#   - Best model  : highest mean R across all tasks and channels
#   - Best mode   : original vs derivative always compared (main contribution)
#   - Best channel: highest mean R across all tasks and models
#   - Worst channel: lowest mean R (for contrast)
#   - Best task   : highest mean R across all channels and models
#
# Output per subject (~10 figures):
#   fig1_r_bar_comparison_{task}.png     — 3 figures, both modes side-by-side
#   fig2_timeseries_best_{ch}.png        — best channel, both modes overlaid
#   fig2_timeseries_worst_{ch}.png       — worst channel, both modes overlaid
#   fig3_scatter_{best_ch}.png           — best model, best channel, both modes
#   fig3_scatter_{worst_ch}.png          — best model, worst channel, both modes
#   fig4_distance_vs_R.png               — both modes
#   selection_log.txt                    — auto-selection summary
# =============================================================================

import matplotlib.patheffects as _pe

PAPER_DPI    = 300
PAPER_DIR    = os.path.join(OUTPUT_DIR, 'paper_figures')
MODE_DISPLAY = {'original': 'Actual Signal',
                'derivative': 'Actual + Derivative Signal'}
MODE_COLOR   = {'original': '#185FA5', 'derivative': '#D85A30'}
MODE_LS      = {'original': '-', 'derivative': '--'}
SPLIT_KEY    = 'split_2'   # last LOTO split


# -----------------------------------------------------------------------------
# Helper: load averaged CSV for a subject
# -----------------------------------------------------------------------------
def _load_avg_csv(subject_id, input_mode):
    label = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    path  = os.path.join(OUTPUT_DIR,
                         f"results_{label}_sub{subject_id}_averaged.csv")
    if not os.path.exists(path):
        print(f"  [WARN] Not found: {path}")
        return None
    return pd.read_csv(path)


# -----------------------------------------------------------------------------
# Helper: auto-select best/worst combinations from averaged results
# -----------------------------------------------------------------------------
def _auto_select(df_orig, df_deriv):
    """
    Returns a dict with selected best model, best/worst channel, best task,
    and best mode based on mean Pearson R.
    """
    sel = {}

    # Combine both modes for cross-mode comparisons
    dfs = []
    if df_orig  is not None: dfs.append(df_orig.assign(input_mode='original'))
    if df_deriv is not None: dfs.append(df_deriv.assign(input_mode='derivative'))
    df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    if df_all.empty:
        return sel

    # Best model: highest mean R across all tasks, channels, modes
    model_r = df_all.groupby('Model')['R'].mean()
    sel['best_model']  = model_r.idxmax()
    sel['model_r_mean'] = round(model_r.max(), 4)

    # Best/worst channel: mean R across all tasks, models, modes
    ch_r = df_all.groupby('Target_CH')['R'].mean()
    sel['best_ch']   = ch_r.idxmax()
    sel['worst_ch']  = ch_r.idxmin()
    sel['best_ch_r'] = round(ch_r.max(), 4)
    sel['worst_ch_r']= round(ch_r.min(), 4)

    # Best task: mean R across all channels, models, modes
    task_r = df_all.groupby('Task')['R'].mean()
    sel['best_task']   = task_r.idxmax()
    sel['task_r_mean'] = round(task_r.max(), 4)

    # Best mode: which input mode gives higher overall mean R
    mode_r = df_all.groupby('input_mode')['R'].mean()
    sel['best_mode']   = mode_r.idxmax()
    sel['mode_r_orig'] = round(mode_r.get('original',   0), 4)
    sel['mode_r_deriv']= round(mode_r.get('derivative', 0), 4)

    return sel


# -----------------------------------------------------------------------------
# Fig 1: R bar comparison — original vs derivative side-by-side, per task
#         3 figures (one per task)
# -----------------------------------------------------------------------------
def _fig1_r_bar_comparison(df_orig, df_deriv, subject_id, viz_dir):
    channels   = list(NEW_ESTIMATION_SCHEME.keys())
    model_list = list(MODEL_COLORS.keys())
    n_ch       = len(channels)
    n_models   = len(model_list)
    bar_width  = 0.09
    x          = np.arange(n_ch)
    saved      = []

    for task in TASKS:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
        fig.patch.set_facecolor('#FAFAFA')
        fig.suptitle(
            f'Pearson R — Original vs Derivative Input | '            f'Task: {task.replace("_", " ")} | Subject: {subject_id}',
            fontsize=12, fontweight='bold'
        )

        for ax, (mode, df) in zip(axes,
                [('original', df_orig), ('derivative', df_deriv)]):
            if df is None:
                ax.set_visible(False)
                continue

            task_df = df[df['Task'] == task]

            for i, model_name in enumerate(model_list):
                r_values = []
                for ch in channels:
                    r = task_df[
                        (task_df['Target_CH'] == ch) &
                        (task_df['Model']     == model_name)
                    ]['R'].mean()
                    r_values.append(r if not np.isnan(r) else 0)

                offset = (i - n_models / 2 + 0.5) * bar_width
                bars   = ax.bar(
                    x + offset, r_values, bar_width,
                    label=model_name,
                    color=MODEL_COLORS[model_name],
                    alpha=0.85, edgecolor='white', linewidth=0.5
                )
                for bar, r in zip(bars, r_values):
                    if r > 0.05:
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.01,
                                f'{r:.2f}', ha='center', va='bottom',
                                fontsize=6, rotation=90)

            ax.axhline(0.7, color='#AAAAAA', linewidth=1.2,
                       linestyle='--', alpha=0.7, label='R = 0.7')
            ax.set_xticks(x)
            ax.set_xticklabels(channels, fontsize=10)
            ax.set_ylim(0, 1.18)
            ax.set_ylabel('Pearson R', fontsize=11)
            ax.set_xlabel('Target Channel', fontsize=11)
            ax.set_title(MODE_DISPLAY[mode], fontsize=10, fontweight='bold',
                         color=MODE_COLOR[mode])
            ax.set_facecolor('#F8F8F8')
            ax.grid(True, axis='y', linestyle='--', alpha=0.35)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.legend(fontsize=8, loc='upper right', framealpha=0.88)

        plt.tight_layout()
        os.makedirs(viz_dir, exist_ok=True)
        fname = f"fig1_r_bar_comparison_{task}.png"
        fpath = os.path.join(viz_dir, fname)
        plt.savefig(fpath, dpi=PAPER_DPI, bbox_inches='tight',
                    facecolor='white')
        plt.close(fig)
        print(f"  [PAPER] Saved: {fname}")
        saved.append(fpath)

    return saved


# -----------------------------------------------------------------------------
# Fig 2: Timeseries — best and worst channel, both modes overlaid
#         2 figures (best_ch, worst_ch)
# -----------------------------------------------------------------------------
def _fig2_timeseries(cache_orig, cache_deriv, subject_id,
                     best_ch, worst_ch, best_task, viz_dir,
                     n_display=3200):
    saved = []

    for ch_label, target_ch in [('best', best_ch), ('worst', worst_ch)]:
        fig, ax = plt.subplots(figsize=(16, 4))
        fig.patch.set_facecolor('#FAFAFA')

        any_plotted = False
        for mode, cache in [('original', cache_orig),
                             ('derivative', cache_deriv)]:
            if cache is None:
                continue
            ch_data = (cache.get(best_task, {})
                            .get(SPLIT_KEY, {})
                            .get(target_ch))
            if ch_data is None:
                continue

            y_true      = ch_data['y_test_orig']
            predictions = ch_data['predictions']
            n_pts       = min(n_display, len(y_true))
            time_axis   = np.arange(n_pts) / SAMPLING_RATE

            if not any_plotted:
                duration = time_axis[-1]
                for sec in range(0, int(duration) + 1, 2):
                    ax.axvspan(sec, min(sec + 1, duration),
                               color='#E8EAF6', alpha=0.35, zorder=1)
                # plot actual once (same signal)
                ax.plot(time_axis, y_true[:n_pts], color='#1A1A2E',
                        linewidth=1.0, alpha=0.9, label='Actual', zorder=6)
                any_plotted = True

            # overlay best model prediction per mode
            best_model_key = max(
                predictions,
                key=lambda m: compute_metrics(y_true, predictions[m])['R']
            )
            y_pred = predictions[best_model_key]
            ax.plot(time_axis, y_pred[:n_pts],
                    color=MODE_COLOR[mode],
                    linewidth=0.9, alpha=0.8,
                    linestyle=MODE_LS[mode],
                    label=f'{MODE_DISPLAY[mode]} ({best_model_key})',
                    zorder=5)

        if not any_plotted:
            plt.close(fig)
            continue

        ax.set_xlim(0, duration)
        ax.set_xticks(np.arange(0, duration + 1, 2))
        ax.set_xlabel('Time (s)', fontsize=11)
        ax.set_ylabel('Amplitude (\u00b5V)', fontsize=11)
        ax.set_title(
            f'Timeseries — Channel: {target_ch} ({ch_label}) | '            f'Task: {best_task.replace("_", " ")} | Subject: {subject_id}\n'            f'Estimated: best model per mode | Split: trial 3 as test (representative)',
            fontsize=11, fontweight='bold'
        )
        ax.set_facecolor('#F4F6FF')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(fontsize=9, loc='upper right',
                  framealpha=0.88, ncol=3)

        plt.tight_layout()
        os.makedirs(viz_dir, exist_ok=True)
        fname = f"fig2_timeseries_{ch_label}_{target_ch}.png"
        fpath = os.path.join(viz_dir, fname)
        plt.savefig(fpath, dpi=PAPER_DPI, bbox_inches='tight',
                    facecolor='white')
        plt.close(fig)
        print(f"  [PAPER] Saved: {fname}")
        saved.append(fpath)

    return saved


# -----------------------------------------------------------------------------
# Fig 3: Scatter — best model, best+worst channel, both modes as subplots
#         2 figures (best_ch, worst_ch), each has 2 subplots (original|deriv)
# -----------------------------------------------------------------------------
def _fig3_scatter(cache_orig, cache_deriv, subject_id,
                  best_model, best_ch, worst_ch, best_task, viz_dir):
    saved = []

    for ch_label, target_ch in [('best', best_ch), ('worst', worst_ch)]:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.patch.set_facecolor('#FAFAFA')
        fig.suptitle(
            f'Scatter — Model: {best_model} | Channel: {target_ch} ({ch_label}) | '            f'Task: {best_task.replace("_"," ")} | Subject: {subject_id}\\n'            f'Split: trial 3 as test (representative)',
            fontsize=11, fontweight='bold'
        )

        for ax, (mode, cache) in zip(axes,
                [('original', cache_orig), ('derivative', cache_deriv)]):
            if cache is None:
                ax.set_visible(False)
                continue

            ch_data = (cache.get(best_task, {})
                            .get(SPLIT_KEY, {})
                            .get(target_ch))
            if ch_data is None:
                ax.set_visible(False)
                continue

            y_true = ch_data['y_test_orig']
            y_pred = ch_data['predictions'].get(best_model)
            if y_pred is None:
                ax.set_visible(False)
                continue

            metrics = compute_metrics(y_true, y_pred)
            color   = MODE_COLOR[mode]

            ax.scatter(y_true, y_pred, color=color,
                       alpha=0.25, s=5, edgecolors='none', zorder=3)

            val_min = min(y_true.min(), y_pred.min())
            val_max = max(y_true.max(), y_pred.max())
            ax.plot([val_min, val_max], [val_min, val_max],
                    color='#1A1A2E', linewidth=1.5,
                    linestyle='--', alpha=0.8, zorder=4)

            metrics_text = (
                f"R    = {metrics['R']:.4f}\n"
                f"RMSE = {metrics['RMSE']:.4f}\n"
                f"MAE  = {metrics['MAE']:.4f}\n"
                f"NSE  = {metrics['NSE']:.4f}"
            )
            ax.text(0.04, 0.97, metrics_text,
                    transform=ax.transAxes, fontsize=9,
                    verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                              alpha=0.88, edgecolor=color, linewidth=1.2))

            ax.set_xlabel('Actual (\u00b5V)', fontsize=11)
            ax.set_ylabel('Estimated (\u00b5V)', fontsize=11)
            ax.set_title(MODE_DISPLAY[mode], fontsize=10,
                         fontweight='bold', color=color)
            ax.set_facecolor('#F8F8F8')
            ax.grid(True, linestyle='--', alpha=0.35, color='#CCCCCC')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_aspect('equal', adjustable='box')

        plt.tight_layout()
        os.makedirs(viz_dir, exist_ok=True)
        fname = f"fig3_scatter_{ch_label}_{target_ch}.png"
        fpath = os.path.join(viz_dir, fname)
        plt.savefig(fpath, dpi=PAPER_DPI, bbox_inches='tight',
                    facecolor='white')
        plt.close(fig)
        print(f"  [PAPER] Saved: {fname}")
        saved.append(fpath)

    return saved


# -----------------------------------------------------------------------------
# Fig 4: Distance vs R — both modes (reuses plot_distance_vs_r)
# -----------------------------------------------------------------------------
def _fig4_distance(df_orig, df_deriv, subject_id, viz_dir):
    return plot_distance_vs_r(
        subject_id=subject_id,
        results_orig=df_orig,
        results_deriv=df_deriv,
        save_dir=viz_dir
    )


# -----------------------------------------------------------------------------
# Main: generate all paper figures for a list of subjects
# -----------------------------------------------------------------------------
def generate_paper_figures(subject_ids=None):
    """
    Auto-select best combinations per subject and generate paper figures.
    Reads from averaged CSVs and training cache pkl files.

    Parameters:
        subject_ids : list of subject IDs to process (default: SUBJECT_IDS)
    """
    if subject_ids is None:
        subject_ids = SUBJECT_IDS

    for subject_id in subject_ids:
        print(f"\n{'='*60}")
        print(f"  Paper Figures | Subject: {subject_id}")
        print(f"{'='*60}")

        viz_dir = os.path.join(PAPER_DIR, f'sub{subject_id}')
        os.makedirs(viz_dir, exist_ok=True)

        # Load averaged CSVs
        df_orig  = _load_avg_csv(subject_id, 'original')
        df_deriv = _load_avg_csv(subject_id, 'derivative')

        if df_orig is None and df_deriv is None:
            print(f"  [SKIP] No averaged CSV found for sub{subject_id}")
            continue

        # Load cache pkl files for raw signal data
        cache_orig  = load_training_results(subject_id, 'original')
        cache_deriv = load_training_results(subject_id, 'derivative')

        # Auto-select best combinations
        sel = _auto_select(df_orig, df_deriv)

        # Print and save selection log
        log_lines = [
            f"Auto-selection log — Subject {subject_id}",
            f"{'='*50}",
            f"Best model   : {sel.get('best_model')}  (mean R = {sel.get('model_r_mean')})",
            f"Best channel : {sel.get('best_ch')}  (mean R = {sel.get('best_ch_r')})",
            f"Worst channel: {sel.get('worst_ch')}  (mean R = {sel.get('worst_ch_r')})",
            f"Best task    : {sel.get('best_task')}  (mean R = {sel.get('task_r_mean')})",
            f"Best mode    : {sel.get('best_mode')}",
            f"  original mean R   = {sel.get('mode_r_orig')}",
            f"  derivative mean R = {sel.get('mode_r_deriv')}",
            f"{'='*50}",
            f"Note: both modes always shown for comparison (main contribution).",
            f"Timeseries & scatter: split_2 (trial 3 as test) — representative split.",
            f"R bar & distance: averaged across all 3 LOTO splits.",
        ]
        for line in log_lines:
            print(f"  {line}")

        log_path = os.path.join(viz_dir, 'selection_log.txt')
        with open(log_path, 'w') as f:
            f.write('\n'.join(log_lines))

        best_model = sel.get('best_model', list(MODEL_COLORS.keys())[0])
        best_ch    = sel.get('best_ch',   list(NEW_ESTIMATION_SCHEME.keys())[0])
        worst_ch   = sel.get('worst_ch',  list(NEW_ESTIMATION_SCHEME.keys())[-1])
        best_task  = sel.get('best_task', TASKS[0])

        # Generate figures
        print(f"\n  Generating figures...")
        _fig1_r_bar_comparison(df_orig, df_deriv, subject_id, viz_dir)
        _fig2_timeseries(cache_orig, cache_deriv, subject_id,
                         best_ch, worst_ch, best_task, viz_dir)
        _fig3_scatter(cache_orig, cache_deriv, subject_id,
                      best_model, best_ch, worst_ch, best_task, viz_dir)
        _fig4_distance(df_orig, df_deriv, subject_id, viz_dir)

        n_figs = len([f for f in os.listdir(viz_dir) if f.endswith('.png')])
        print(f"\n  Done — {n_figs} paper figures saved in: {viz_dir}")


# =============================================================================
# Run — generate paper figures for all subjects
# =============================================================================
# generate_paper_figures(subject_ids=SUBJECT_IDS)
# Uncomment the line above to run. Requires training to be completed first.
# Can also specify a subset: generate_paper_figures(subject_ids=[1, 2])
